### 🚀 Run this notebook in [RenkuLab](https://renkulab.io/)

👉 [![launch - renku](https://renkulab.io/renku-badge.svg)](https://renkulab.io/p/meteoswiss/opendata-nwp-demos/sessions/01KME52HC2FZ6ZHB30SSFG08PW/start)

1. Wait for the session to start
2. Navigate to `opendata-nwp-demos/09_constant_parameters.ipynb` and run the notebook without any further installation.

# Accessing constant parameters

This notebook demonstrates the retrieval of **constant run parameters** such as grid definition, vertical layers, and surface descriptors that are not part of the forecast GRIB files, and shows how to check that they match the forecast parameters. The data is provided by MeteoSwiss as part of Switzerland’s [Open Government Data (OGD) initiative](https://www.meteoswiss.admin.ch/services-and-publications/service/open-data.html).



---

## 🔍 **What You’ll Do in This Notebook**

 📥  **Retrieve vertical constants**  
    The parameter HHL is the only vertical constant; retrieval is shown for ICON-CH1-EPS and ICON-CH2-EPS.
    
| Name      | Description                                          |
|-----------|------------------------------------------------------|
| HHL       | Geometric Height of the layer limits above sea level, see the [model grid documentation](https://opendatadocs.meteoswiss.ch/e-forecast-data/e2-e3-numerical-weather-forecasting-model#vertical-grid) |

 📥  **Retrieve horizontal constants**  
    The horizontal constant parameters are CLON, CLAT, FR_LAND, HSURF, and SOILTYP; you'll retrieve them for ICON-CH2-EPS.
    
| Name      | Description                                                                      |
|-----------|----------------------------------------------------------------------------------|
| CLON      | Longitude of the center point coordinate of each triangle on the horizontal grid |
| CLAT      | Latitude of the center point coordinate of each triangle on the horizontal grid  |
| FR_LAND   | Land cover indicator (1 for land, 0 for sea)                                     |
| HSURF     | Geometric height of the earth's surface above sea level                          |
| SOILTYPE  | Type of soil, based on the [soilType.table](https://github.com/COSMO-ORG/eccodes-cosmo-resources/blob/master/definitions/soilType.table) |

 ✅  **Verify**  
    Using GRIB metadata (uuidOfHGrid), you'll verify grid consistency with a forecast parameter from ICON-CH2-EPS.
    
 📈  **Plot**  
    Create a profile of the vertical wind speed at Zurich airport, and show mean precipitation as a function of surface height.

---

> ⚠️ **Warning**: The constant parameters are only considered constant per run! Although they are not expected to change often, we recommend that you check that the 'uuidOfHGrid' agrees with the forecast data, especially if you cache the constants.

> ⚠️ **Warning**: At the moment, the collection assets are not versioned such that there is no way to fetch a previous version of the constant run parameters. If you are building an archive, you may need to keep track of those constants.

## 📥 Retrieve constant parameters
The constant parameters are separated into horizontal and vertical constants, and available for both the ICON-CH1-EPS and ICON-CH2-EPS collections.

First, the urls are obtained:

In [ ]:
from meteodatalab import ogd_api

# Vertical
url_ch1_vert = ogd_api.get_collection_asset_url(
    collection_id="ch.meteoschweiz.ogd-forecasting-icon-ch1",
    asset_id="vertical_constants_icon-ch1-eps.grib2"
)
url_ch2_vert = ogd_api.get_collection_asset_url(
    collection_id="ch.meteoschweiz.ogd-forecasting-icon-ch2",
    asset_id="vertical_constants_icon-ch2-eps.grib2"
)

# Horizontal
url_ch1_hor = ogd_api.get_collection_asset_url(
    collection_id="ch.meteoschweiz.ogd-forecasting-icon-ch1",
    asset_id="horizontal_constants_icon-ch1-eps.grib2"
)
url_ch2_hor = ogd_api.get_collection_asset_url(
    collection_id="ch.meteoschweiz.ogd-forecasting-icon-ch2",
    asset_id="horizontal_constants_icon-ch2-eps.grib2"
)

Next, the data can be fetched. HHL is the only vertical constant, referring to the geometric height of the layer limits above sea level. For more information, see the [model grid documentation](https://opendatadocs.meteoswiss.ch/e-forecast-data/e2-e3-numerical-weather-forecasting-model#vertical-grid).

> 💡 **Tip**: The HHL parameter is also used in the notebook [05_interpolate_vertically.ipynb](05_interpolate_vertically.ipynb).

In [ ]:
from meteodatalab import grib_decoder, data_source

# load CH2 vertical constants
ds_ch2_vert = grib_decoder.load(
    source=data_source.URLDataSource(urls=[url_ch2_vert]),
    request={"param": "HHL"},
    geo_coords=lambda uuid: {}
)

# load CH1 vertical constants
ds_ch1_vert = grib_decoder.load(
    source=data_source.URLDataSource(urls=[url_ch1_vert]),
    request={"param": "HHL"},
    geo_coords=lambda uuid: {}
)

ds_ch1_vert["HHL"]

In the horizontal constants, CLON, CLAT, FR_LAND, HSURF and SOILTYP are available:
| Name      | Description                                                                      |
|-----------|----------------------------------------------------------------------------------|
| CLON      | Longitude of the center point coordinate of each triangle on the horizontal grid |
| CLAT      | Latitude of the center point coordinate of each triangle on the horizontal grid  |
| FR_LAND   | Land cover indicator (1 for land, 0 for sea)                                     |
| HSURF     | Geometric height of the earth's surface above sea level                          |
| SOILTYPE  | Type of soil, based on the [soilType.table](https://github.com/COSMO-ORG/eccodes-cosmo-resources/blob/master/definitions/soilType.table)                                                                |

In [ ]:
# load CH2 horizontal constants
ds_ch2_hor = grib_decoder.load(
    source=data_source.URLDataSource(urls=[url_ch2_hor]),
    request={"param": ["CLON", "CLAT", "FR_LAND", "HSURF", "SOILTYP"]},
    geo_coords=lambda uuid: {}
)

In [ ]:
ds_ch2_hor.keys()

## ✅ Verify grid consistency

The grib metadata carries a lot of information:

In [ ]:
# show all metadata
ds_ch2_vert["HHL"].metadata.dump()

A universally unique identifier (UUID) of the horizontal grid the data is associated with is provided. You can see that the ICON-CH1-EPS and ICON-CH2-EPS models use a different grid:

In [ ]:
print("CH1-EPS HHL    : ", ds_ch1_vert["HHL"].metadata.get("uuidOfHGrid"))
print("CH2-EPS HHL    : ", ds_ch2_vert["HHL"].metadata.get("uuidOfHGrid"))
print("CH2-EPS SOILTYP: ", ds_ch2_hor["SOILTYP"].metadata.get("uuidOfHGrid"))

We fetch now the vertical wind speed from the ICON-CH2-EPS control forecast, and compare the UUID of the grid with the UUID obtained from the vertical constants.

> ⚠️ **Warning**: The constant parameters are only considered constant per run! Although they are not expected to change often, we recommend that you check that the 'uuidOfHGrid' agrees with the forecast data, especially if you cache the constants.

In [ ]:
from meteodatalab import ogd_api
from earthkit.data import config
config.set("cache-policy", "temporary")

da_W_ch2 = ogd_api.get_from_ogd(
    ogd_api.Request(
        collection="ogd-forecasting-icon-ch2",
        variable="W",
        ref_time="latest",
        perturbed=False,
        lead_time=f"P4DT00H",
    )
)

Compare the UUIDs:

In [ ]:
da_HHL_ch2 = ds_ch2_vert["HHL"]
assert da_W_ch2.metadata.get("uuidOfHGrid") == da_HHL_ch2.metadata.get("uuidOfHGrid")

## 📈 Create a profile of the vertical wind speed

Now that we have verified consistency of the constant run parameters (HHL) and forecast data (W), we can create a plot.

In [ ]:
from earthkit.geo import nearest_point_haversine

# target coordinates
zrh_geo_point = (47.453928, 8.565074)  # zrh airport

# get index of closest cell
p = nearest_point_haversine(zrh_geo_point, (da_W_ch2.lat.values, da_W_ch2.lon.values))[0][0]

# extract the data to plot at index p
w_profile = da_W_ch2.isel(cell=p, eps=0, ref_time=0, lead_time=0)
h_profile = da_HHL_ch2.isel(cell=p, eps=0, ref_time=0, lead_time=0)

In [ ]:
from earthkit.plots.interactive import Chart

# create plot

chart = Chart()

chart.line(x=w_profile, y=h_profile)

chart.fig.update_layout(xaxis1={"title": f"Vertical Wind ({da_W_ch2.metadata.get("units")})"})
chart.fig.update_layout(yaxis1={"title": f"Height above sea level ({da_HHL_ch2.metadata.get("units")})"})

chart.title(f"Vertical Profile at Zurich Airport: Vertical Wind")
chart.fig.update_layout(title_font=dict(size=12),font=dict(size=12))

chart.show("png")


## 📈 Show the precipitation amounts against surface height

In [ ]:
# fetch total precipitation for 5 days lead time
from meteodatalab import ogd_api
from earthkit.data import config
config.set("cache-policy", "temporary")

da_TOT_PREC_ch2 = ogd_api.get_from_ogd(
    ogd_api.Request(
        collection="ogd-forecasting-icon-ch2",
        variable="TOT_PREC",
        ref_time="latest",
        perturbed=False,
        lead_time=f"P5DT00H",
    )
)

In [ ]:
# verify consistency of the grid
da_HSURF_ch2 = ds_ch2_hor["HSURF"]
da_HHL_ch2 = ds_ch2_vert["HHL"]
assert da_W_ch2.metadata.get("uuidOfHGrid") == da_HSURF_ch2.metadata.get("uuidOfHGrid")

In [ ]:
import numpy as np
import pandas as pd
from earthkit.plots.interactive import Chart

# squeeze to 1D arrays over cells
prec_vals = da_TOT_PREC_ch2.squeeze().values
hsurf_vals = da_HSURF_ch2.squeeze().values

# bin cells by surface height and compute mean precipitation per bin
n_bins = 20
bins = np.linspace(hsurf_vals.min(), hsurf_vals.max(), n_bins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

df = pd.DataFrame({"prec": prec_vals, "hsurf": hsurf_vals})
df["bin"] = pd.cut(df["hsurf"], bins=bins, labels=bin_centers, include_lowest=True)
mean_prec = df.groupby("bin", observed=True)["prec"].mean()

# create plot
chart = Chart()
chart.line(x=mean_prec.index.astype(float).values, y=mean_prec.values)
chart.fig.update_layout(
    xaxis1={"title": f"Surface Height ({da_HSURF_ch2.metadata.get('units')})"},
    yaxis1={"title": f"Mean Precipitation ({da_TOT_PREC_ch2.metadata.get('units')})"},
)

ref_time = pd.Timestamp(da_TOT_PREC_ch2.ref_time.values[0]).strftime("%Y-%m-%d %H:%M UTC")
lead_time = pd.Timedelta(da_TOT_PREC_ch2.lead_time.values[0])
lead_hours = int(lead_time.total_seconds() / 3600)

chart.title(f"Mean Precipitation vs Surface Height, {ref_time} +{lead_hours}h")
chart.fig.update_layout(title_font=dict(size=12), font=dict(size=12))
chart.show("png")
